# 🎬 Movie Recommendation System
### Fixed & Improved Version — with Pickle Export for FastAPI Backend

**Run all cells top to bottom.**  
At the end, 4 pickle files will be saved:
- `df.pkl` — cleaned DataFrame
- `indices.pkl` — title → row index map
- `tfidf_matrix.pkl` — sparse TF-IDF matrix
- `tfidf.pkl` — fitted TfidfVectorizer

Copy those 4 files into the `backend/` folder before running the FastAPI server.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 📥 Load Data

In [ ]:
df = pd.read_csv('/content/movies_metadata.csv')
print(f"Loaded {len(df)} rows")
df.head()

## 🧹 Data Cleaning

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]

df = df.dropna(subset=['title'])

df['overview'] = df['overview'].fillna('')
df['tagline']  = df['tagline'].fillna('')

df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce').fillna(0)
df['popularity']   = pd.to_numeric(df['popularity'],   errors='coerce').fillna(0)

print(df.isnull().sum())
df.head()

## 🎭 Genre Parsing

In [ ]:
import ast

def parse_genres(x):
    try:
        return " ".join([i['name'] for i in ast.literal_eval(x)])
    except (ValueError, TypeError):
        return ''

df['genres'] = df['genres'].apply(parse_genres)
df.head()

## 🧠 Feature Engineering
Genres repeated 3x to boost their TF-IDF weight over random overview words.

In [ ]:
df['final_features'] = (
    df['overview'] + " " +
    df['genres']   + " " + df['genres'] + " " + df['genres'] + " " +
    df['tagline']
)
print(df['final_features'][0])

## ✂️ Text Preprocessing

In [ ]:
import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def preprocess_text(text):
    text  = str(text).lower()
    text  = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

df['final_features'] = df['final_features'].apply(preprocess_text)
print("Preprocessing done.")

## 🔢 Build Index Map

In [ ]:
df = df.reset_index(drop=True)

indices = pd.Series(df.index, index=df['title'])
indices = indices[~indices.index.duplicated(keep='first')]

print(f"Unique titles in index: {len(indices)}")
indices.head()

## 🔡 TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['final_features'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## 🎯 Local Recommendation Function

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def what_to_watch_now(title, n=10, use_reranking=True):
    if title not in indices:
        print(f"❌ '{title}' not found in dataset.")
        return None

    idx = indices[title]
    if hasattr(idx, '__len__'):
        idx = int(idx.iloc[0])

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    candidate_count = n * 3
    similar_idx = sim_scores.argsort()[::-1][1:candidate_count + 1]

    results = df.iloc[similar_idx][['title', 'genres', 'vote_average', 'popularity']].copy()
    results['similarity'] = sim_scores[similar_idx]

    if use_reranking:
        results['vote_norm'] = results['vote_average'] / 10.0
        pop_max = df['popularity'].max()
        results['pop_norm'] = results['popularity'] / pop_max if pop_max > 0 else 0
        results['final_score'] = (
            0.60 * results['similarity'] +
            0.30 * results['vote_norm'] +
            0.10 * results['pop_norm']
        )
        results = results.sort_values('final_score', ascending=False)
        return results[['title', 'genres', 'vote_average', 'similarity', 'final_score']].head(n).reset_index(drop=True)
    else:
        return results[['title', 'genres', 'vote_average', 'similarity']].head(n).reset_index(drop=True)

## 🧪 Test the Recommender

In [ ]:
results = what_to_watch_now('The Dark Knight', n=10)
if results is not None:
    display(results)

## 💾 Export Pickles for FastAPI Backend
Run this cell last. Copy the 4 `.pkl` files into your `backend/` folder.

In [ ]:
import pickle, os

EXPORT_DIR = '/content'
os.makedirs(EXPORT_DIR, exist_ok=True)

artifacts = {
    'df.pkl':           df,
    'indices.pkl':      indices,
    'tfidf_matrix.pkl': tfidf_matrix,
    'tfidf.pkl':        tfidf,
}

for fname, obj in artifacts.items():
    path = os.path.join(EXPORT_DIR, fname)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"✅ Saved {fname} ({size_mb:.1f} MB)")

print("\n🎉 Done! Copy these 4 files into your backend/ folder.")

---
## 👨‍💻 Author

**Mukund Chaurasiya**  
Developer · ML Enthusiast · Open Source Contributor

| | |
|---|---|
| 📧 Email | [chaurasiyamukund2006@gmail.com](mailto:chaurasiyamukund2006@gmail.com) |
| 🔗 LinkedIn | [linkedin.com/in/mukund-chaurasiya-11a872385](https://www.linkedin.com/in/mukund-chaurasiya-11a872385/) |
| 🐙 GitHub | [github.com/mukundpy](https://github.com/mukundpy) |